In [1]:
import pandas as pd
import numpy as np
import pickle
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load credit card data
df = pd.read_csv("../data/creditcard.csv")
print("Original shape:", df.shape)

Original shape: (284807, 31)


In [2]:
scaler = StandardScaler()
df['scaled_amount'] = scaler.fit_transform(df['Amount'].values.reshape(-1, 1))
df['scaled_time'] = scaler.fit_transform(df['Time'].values.reshape(-1, 1))

# Drop the original Time and Amount
df = df.drop(['Time', 'Amount'], axis=1)

print("New shape:", df.shape)
print("Columns:", df.columns.tolist()[-5:])  # show last 5

New shape: (284807, 31)
Columns: ['V27', 'V28', 'Class', 'scaled_amount', 'scaled_time']


In [3]:
X = df.drop('Class', axis=1)
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)
print("Fraud in train:", y_train.sum())
print("Fraud in test:", y_test.sum())

Train: (227845, 30)
Test: (56962, 30)
Fraud in train: 394
Fraud in test: 98


In [4]:
os.makedirs("../data/clean", exist_ok=True)

with open("../data/clean/cc_data.pkl", "wb") as f:
    pickle.dump((X_train, X_test, y_train, y_test), f)

print("Saved to data/clean/cc_data.pkl")

Saved to data/clean/cc_data.pkl


In [5]:
# Load Home Credit
hc = pd.read_csv("../data/home_credit/application_train.csv")
print("Original shape:", hc.shape)

# Drop columns with >50% missing values
missing_pct = hc.isnull().sum() / len(hc)
cols_to_drop = missing_pct[missing_pct > 0.5].index
hc = hc.drop(columns=cols_to_drop)
print("After dropping high-missing columns:", hc.shape)
print("Dropped", len(cols_to_drop), "columns")

Original shape: (307511, 122)
After dropping high-missing columns: (307511, 81)
Dropped 41 columns


In [6]:
# Fix the DAYS_EMPLOYED anomaly (365243 = unemployed)
hc['DAYS_EMPLOYED'] = hc['DAYS_EMPLOYED'].replace(365243, np.nan)
print("Fixed DAYS_EMPLOYED anomaly")

# Encode categorical columns
from sklearn.preprocessing import LabelEncoder

cat_cols = hc.select_dtypes(include=['object']).columns.tolist()
print(f"Found {len(cat_cols)} categorical columns")

le = LabelEncoder()
for col in cat_cols:
    hc[col] = hc[col].astype(str)
    hc[col] = le.fit_transform(hc[col])

print("Encoded all categorical columns")

Fixed DAYS_EMPLOYED anomaly
Found 13 categorical columns
Encoded all categorical columns


In [7]:
# Fill remaining missing values with median
hc = hc.fillna(hc.median(numeric_only=True))
print("Remaining missing values:", hc.isnull().sum().sum())

# Features and target
X_hc = hc.drop(['TARGET', 'SK_ID_CURR'], axis=1)
y_hc = hc['TARGET']

print("Feature shape:", X_hc.shape)

# Split
X_hc_train, X_hc_test, y_hc_train, y_hc_test = train_test_split(
    X_hc, y_hc, test_size=0.2, random_state=42, stratify=y_hc
)

# Scale
scaler_hc = StandardScaler()
X_hc_train = scaler_hc.fit_transform(X_hc_train)
X_hc_test = scaler_hc.transform(X_hc_test)

# Save
with open("../data/clean/hc_data.pkl", "wb") as f:
    pickle.dump((X_hc_train, X_hc_test, y_hc_train, y_hc_test, X_hc.columns.tolist()), f)

print("Home Credit saved!")
print("Train:", X_hc_train.shape, "| Test:", X_hc_test.shape)

Remaining missing values: 0
Feature shape: (307511, 79)
Home Credit saved!
Train: (246008, 79) | Test: (61503, 79)


In [9]:
import pickle
import os

os.makedirs("../models", exist_ok=True)

with open("../models/hc_scaler.pkl", "wb") as f:
    pickle.dump(scaler_hc, f)

print(" Scaler saved successfully!")

 Scaler saved successfully!


In [11]:
import pickle
# Added ../ to fix the path
with open("../data/clean/cc_data.pkl", "rb") as f:
    X_train, X_test, y_train, y_test = pickle.load(f)

# Find the first actual fraud in the test set
fraud_index = y_test[y_test == 1].index[0]
print(X_test.loc[fraud_index])

V1              -1.271244
V2               2.462675
V3              -2.851395
V4               2.324480
V5              -1.372245
V6              -0.948196
V7              -3.065234
V8               1.166927
V9              -2.268771
V10             -4.881143
V11              2.255147
V12             -4.686387
V13              0.652375
V14             -6.174288
V15              0.594380
V16             -4.849692
V17             -6.536521
V18             -3.119094
V19              1.715494
V20              0.560478
V21              0.652941
V22              0.081931
V23             -0.221348
V24             -0.523582
V25              0.224228
V26              0.756335
V27              0.632800
V28              0.250187
scaled_amount   -0.353189
scaled_time     -0.796134
Name: 77348, dtype: float64
